# glass2: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('19_glass2_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,100.000,50.000,34.663,86.902,0.000
smote,63.947,34.771,26.687,53.333,87.071,70.202,49.934,84.436,49.359
bsmote,57.815,32.993,25.750,48.333,88.064,68.199,45.274,77.838,45.404
ros,66.624,37.071,28.060,57.500,87.551,72.526,50.917,85.838,51.847
adasyn,70.732,40.187,31.520,58.333,88.333,73.333,53.661,84.712,55.460
casb,47.156,27.583,24.119,36.667,90.846,63.756,39.592,82.498,37.370
casb_relaxed,62.239,34.899,27.718,50.000,88.077,69.038,44.886,82.637,48.569
casb_edited,67.108,33.759,25.086,55.000,83.237,69.119,43.810,81.722,50.433
casb_ros,39.164,23.398,21.500,29.167,91.346,60.256,38.806,83.241,31.281


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,98.974,49.487,31.307,72.615,0.000
smote,68.450,33.640,24.520,58.333,81.699,70.016,34.047,81.082,51.045
bsmote,54.818,25.435,18.821,41.667,82.962,62.314,28.633,73.470,40.126
ros,71.191,36.775,26.699,61.667,84.237,72.952,34.246,73.958,53.983
adasyn,68.634,31.790,21.903,60.833,79.436,70.135,35.154,82.640,50.212
casb,61.086,29.275,22.063,45.000,85.263,65.131,28.621,74.110,45.180
casb_relaxed,65.558,32.029,23.647,52.500,82.981,67.740,33.201,80.213,48.793
casb_edited,67.806,31.029,22.036,59.167,79.173,69.170,32.758,80.963,49.417
casb_ros,61.193,29.424,22.115,45.000,85.519,65.260,28.377,72.468,45.309


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,100.000,50.000,42.115,82.303,0.000
smote,47.899,33.333,44.262,30.833,94.141,62.487,46.802,88.417,40.616
bsmote,25.617,22.333,35.000,16.667,94.628,55.647,40.192,84.848,23.975
ros,30.274,23.167,30.833,19.167,95.160,57.163,42.137,86.357,26.720
adasyn,50.002,33.611,39.119,34.167,93.641,63.904,44.747,88.639,41.807
casb,45.919,32.333,47.857,28.333,94.635,61.484,45.022,85.993,39.126
casb_relaxed,47.783,32.207,42.972,31.667,92.865,62.266,42.291,86.588,39.995
casb_edited,46.765,27.048,27.727,31.667,89.551,60.609,41.810,84.562,36.906
casb_ros,30.265,27.048,35.000,23.333,95.660,59.497,45.890,86.517,28.656


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,98.974,49.487,39.458,80.021,0.000
smote,76.760,43.453,32.101,70.833,85.506,78.170,47.569,83.795,60.106
bsmote,67.774,35.692,25.355,64.167,84.526,74.346,47.327,81.378,51.733
ros,79.002,43.233,30.878,76.667,83.994,80.330,43.485,87.479,61.117
adasyn,76.855,44.263,32.736,71.667,86.538,79.103,50.254,86.154,60.559
casb,72.798,37.676,27.209,65.000,84.032,74.516,33.235,81.590,55.237
casb_relaxed,67.615,34.048,23.800,63.333,84.038,73.686,40.144,81.444,50.832
casb_edited,67.941,33.384,22.884,65.833,80.981,73.407,39.687,81.927,50.663
casb_ros,75.213,39.192,27.901,70.000,83.532,76.766,32.906,82.389,57.202


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,99.487,49.744,36.886,80.460,0.000
smote,64.264,36.299,31.892,53.333,87.104,70.219,44.588,84.432,50.282
bsmote,51.506,29.113,26.232,42.708,87.545,65.127,40.357,79.384,40.310
ros,61.773,35.061,29.118,53.750,87.736,70.743,42.696,83.408,48.417
adasyn,66.556,37.463,31.319,56.250,86.987,71.619,45.954,85.536,52.009
casb,56.740,31.717,30.312,43.750,88.694,66.222,36.618,81.048,44.228
casb_relaxed,60.798,33.296,29.534,49.375,86.990,68.183,40.131,82.720,47.047
casb_edited,62.405,31.305,24.433,52.917,83.236,68.076,39.516,82.294,46.855
casb_ros,51.459,29.766,26.629,41.875,89.014,65.445,36.495,81.154,40.612


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,67.108,adasyn,70.732,-3.624,LOSS
1,f1,casb_relaxed,34.899,adasyn,40.187,-5.288,LOSS
2,precision,casb_relaxed,27.718,adasyn,31.520,-3.802,LOSS
3,recall,casb_edited,55.000,adasyn,58.333,-3.333,LOSS


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,67.806,ros,71.191,-3.385,LOSS
1,f1,casb_relaxed,32.029,ros,36.775,-4.745,LOSS
2,precision,casb_relaxed,23.647,ros,26.699,-3.052,LOSS
3,recall,casb_edited,59.167,ros,61.667,-2.500,LOSS


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,47.783,adasyn,50.002,-2.220,LOSS
1,f1,casb,32.333,adasyn,33.611,-1.278,LOSS
2,precision,casb,47.857,smote,44.262,3.595,WIN
3,recall,casb_relaxed,31.667,adasyn,34.167,-2.500,LOSS


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,62.405,adasyn,66.556,-4.151,LOSS
1,f1,casb_relaxed,33.296,adasyn,37.463,-4.167,LOSS
2,precision,casb,30.312,smote,31.892,-1.580,LOSS
3,recall,casb_edited,52.917,adasyn,56.250,-3.333,LOSS


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.000000,0.000000,0.000000,0.000000,1.000000,0.500000,0.283183,0.814103,0.000000,None
1,42,1,svm,smote,0.50,0.679366,0.444444,0.400000,0.500000,0.923077,0.711538,0.381385,0.897436,0.561905,None
2,42,1,svm,bsmote,0.50,0.000000,0.000000,0.000000,0.000000,0.897436,0.448718,0.221654,0.788462,0.000000,None
3,42,1,svm,ros,0.50,0.000000,0.000000,0.000000,0.000000,0.897436,0.448718,0.338194,0.884615,0.000000,None
4,42,1,svm,adasyn,1.00,0.820413,0.545455,0.428571,0.750000,0.897436,0.823718,0.511111,0.935897,0.682934,None
5,42,1,svm,casb,1.00,0.820413,0.545455,0.428571,0.750000,0.897436,0.823718,0.486111,0.929487,0.682934,None
6,42,1,svm,casb_relaxed,0.75,0.679366,0.444444,0.400000,0.500000,0.923077,0.711538,0.440476,0.923077,0.561905,None
7,42,1,svm,casb_edited,0.50,0.669864,0.400000,0.333333,0.500000,0.897436,0.698718,0.295833,0.833333,0.534932,None
8,42,1,svm,casb_ros,0.75,0.000000,0.000000,0.000000,0.000000,0.897436,0.448718,0.351587,0.891026,0.000000,None
9,42,1,knn,original,0.50,0.000000,0.000000,0.000000,0.000000,0.948718,0.474359,0.167940,0.634615,0.000000,None
